# TopoGen Earth — Interactive Mask-to-Satellite Demo

Draw a land-cover mask with class-colored brushes and generate a satellite image
using a pretrained Flow-Matching UNet checkpoint.

---
## Setup

In [ ]:
!git clone https://github.com/mihalko711/topogen-earth.git
!pip install -q gradio
%cd topogen-earth

## Load model from checkpoint

Fill in `CKPT_PATH` with the path to your `.pth` checkpoint file.

In [ ]:
import sys; sys.path.insert(0, ".")
import torch
from src.models.config import UNetConfig
from src.models.model import create_unet
from src import gradio_app

CKPT_PATH = ""  # <-- FILL IN
assert CKPT_PATH, "Set CKPT_PATH to your checkpoint file"

model_cfg = UNetConfig(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(32, 64, 128),
)
model = create_unet(model_cfg)
ckpt = torch.load(CKPT_PATH, map_location="cuda", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval().to("cuda")

gradio_app.MODEL = model
print(f"Model loaded from epoch {ckpt.get('epoch', '?')}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

## Launch

Opens a public shareable link (valid for ~72h on Kaggle).

In [ ]:
gradio_app.demo.launch(share=True)